# 실시간 스쿼트 자세 교정 AI 시스템

**핵심 기능**
- 대퇴골 길이 정규화 · Visibility 가중 평균 · N프레임 안정화
- descent/bottom 시계열 분리 · Confidence 기반 양방향 보정
- 다중 라벨 분류 · 깊이 부족 자동 감지
- 모던 다크 UI · 한글 폰트 렌더링 · 실시간 카운터 + 폼 스코어
---


### 1. 환경 설정 및 라이브러리 로드

##### 1-1. 라이브러리 로드

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import pickle
import os
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont   # 한글 폰트 렌더링

plt.rcParams['font.family'] = 'Malgun Gothic'

print(f'mediapipe 버전: {mp.__version__}')
print('라이브러리 로드 완료!')


##### 1-2. MediaPipe 초기화

| 모드 | static_image_mode | 설명 |
|------|-------------------|------|
| 이미지 전처리용 | `True` | 매 프레임 독립 처리 |
| 영상 / 웹캠용 | `False` | 이전 프레임 추적 활용 → 더 빠름 |

In [ ]:
# 이미지 전처리용
mp_pose_img = mp.solutions.pose.Pose(
    static_image_mode=True,
    model_complexity=2,
    min_detection_confidence=0.5
)

# 영상 / 웹캠용
mp_pose_video = mp.solutions.pose.Pose(
    static_image_mode=False,
    model_complexity=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

mp_draw = mp.solutions.drawing_utils
print('MediaPipe 초기화 완료!')

---
### 2. 각도 기준값 및 판정 로직 설정

**노이즈 강건 페이즈 감지 (N프레임 연속 조건)**

| 단계 | 조건 | 동작 |
|------|------|------|
| ENTER | 무릎 < 155°가 3프레임 연속 | IDLE → SQUATTING |
| EXIT  | 무릎 ≥ 165°가 3프레임 연속 | SQUATTING → IDLE |

**4단 판정 구조**

```
1) 동작 완수 검증     →  최저 무릎 > 120° → 'shallow'
2) 시계열 구간 분리   →  descent / bottom 평균
3) High-Confidence Good 우선 → good 확률 ≥ 0.55 이면 즉시 good
4) 다중 라벨 + 양방향 보정 → 결함 리스트 산출
```


In [ ]:

# ── 히스테리시스 페이즈 감지 ──
SQUAT_ENTER      = 155
SQUAT_EXIT       = 165
SQUAT_BOTTOM     = 90
MIN_SQUAT_FRAMES = 10
N_CONSEC_FRAMES  = 3

# ── 규칙 보조 판정 ──
# torso 각도: 수직선 대비 몸통 기울기 (클수록 굽음)
TORSO_BENT_THR   = 60      # torso > 60°면 굽음
TORSO_OK_THR     = 45      # torso < 45°면 명백 정상
# hip 각도: 다리-몸통 사이 각도 (작을수록 굽음)
HIP_BENT_THR     = 65
HIP_OK_THR       = 90

# heel_lift: 대퇴골 길이 대비 비율 (양수가 클수록 들림)
HEEL_LIFT_THR    = 0.10    # 측면 촬영 노이즈를 고려해 여유 있게 설정
HEEL_OK_THR      = 0.05    # 이 이하면 명백 정상

# ── 동작 완수 검증 ──
NOT_DEEP_ENOUGH  = 120

# ── 모델 신뢰도 ──
HIGH_CONF_GOOD   = 0.55
MULTI_LABEL_THR  = 0.50    # 다중 라벨로 인정할 최소 확률
CONFIDENCE_THR   = 0.55

# ── 시계열 가중치 ──
W_BOTTOM         = 0.6
W_DESCENT        = 0.4

print('각도 기준값 설정 완료!')
print(f'  ENTER/EXIT       : <{SQUAT_ENTER}° / ≥{SQUAT_EXIT}° ({N_CONSEC_FRAMES}f 연속)')
print(f'  깊이 미달        : 최저 >{NOT_DEEP_ENOUGH}° → shallow')
print(f'  몸통 굽음        : torso > {TORSO_BENT_THR}° AND/OR hip < {HIP_BENT_THR}°')
print(f'  뒤꿈치 들림      : heel_lift > {HEEL_LIFT_THR}')
print(f'  모델 신뢰도      : high-good ≥ {HIGH_CONF_GOOD}, multi-label ≥ {MULTI_LABEL_THR}')


def predict_from_features(feat_list, verbose=False):
    """
    영상 추론 — 모델 예측 우선 + 규칙은 안전망.
    발뒤꿈치 false positive 차단 강화.
    Returns: list[str]
    """
    feat_arr = np.array(feat_list)
    knee_angles = (feat_arr[:, 0] + feat_arr[:, 1]) / 2
    min_knee = float(knee_angles.min())

    # 1) 깊이 검증
    if min_knee > NOT_DEEP_ENOUGH:
        if verbose:
            print(f'[SHALLOW] min_knee={min_knee:.1f}')
        return ['shallow']

    # 2) 시계열 구간 분리
    n = len(feat_arr)
    min_idx = int(np.argmin(knee_angles))

    n_bottom = max(1, int(n * 0.3))
    bottom_idx = np.argsort(knee_angles)[:n_bottom]
    bottom_feat = feat_arr[bottom_idx].mean(axis=0)
    descent_feat = (feat_arr[:min_idx].mean(axis=0)
                    if min_idx > 2 else bottom_feat)

    # 3) 보텀 자세 분석
    hip_angle   = (bottom_feat[2]  + bottom_feat[3])  / 2
    torso_angle = (bottom_feat[10] + bottom_feat[11]) / 2
    heel_lift_L = bottom_feat[8]
    heel_lift_R = bottom_feat[9]

    # 4) 모델 확률
    bottom_proba  = rf_model.predict_proba([bottom_feat])[0]
    descent_proba = rf_model.predict_proba([descent_feat])[0]
    combined      = W_BOTTOM * bottom_proba + W_DESCENT * descent_proba

    classes = list(rf_le.classes_)
    def score(name, proba):
        return proba[classes.index(name)] if name in classes else 0.0

    good_score     = score('good', combined)
    bad_back_score = max(score('bad_back', descent_proba),
                         score('bad_back', bottom_proba))
    bad_heel_score = max(score('bad_heel', bottom_proba),
                         score('bad_heel', descent_proba))

    # 규칙 판정 — heel은 보수적으로
    back_bent = (torso_angle > TORSO_BENT_THR) or (hip_angle < HIP_BENT_THR)
    back_ok   = (torso_angle < TORSO_OK_THR)  and (hip_angle > HIP_OK_THR)
    heel_lifted = (heel_lift_L > HEEL_LIFT_THR) or (heel_lift_R > HEEL_LIFT_THR)
    heel_ok     = (heel_lift_L < HEEL_OK_THR) and (heel_lift_R < HEEL_OK_THR)

    if verbose:
        print(f'[score] good={good_score:.3f}  back={bad_back_score:.3f}  heel={bad_heel_score:.3f}')
        print(f'[angle] hip={hip_angle:.1f}  torso={torso_angle:.1f}  '
              f'heel_L={heel_lift_L:.3f}  heel_R={heel_lift_R:.3f}')
        print(f'[rule]  back_bent={back_bent}  back_ok={back_ok}  '
              f'heel_lifted={heel_lifted}  heel_ok={heel_ok}')

    # ── 5) 판정 ──

    # (a) 모델이 강하게 good → 즉시 good (단, 규칙상 명백 위반은 예외)
    if good_score >= HIGH_CONF_GOOD:
        emergency = []
        if back_bent and bad_back_score > 0.30:
            emergency.append('bad_back')
        # heel은 규칙(heel_lifted) + 모델 점수 둘 다 만족해야 추가
        if heel_lifted and bad_heel_score > 0.40:
            emergency.append('bad_heel')
        result = emergency if emergency else ['good']
        if verbose: print(f'  → HIGH-CONF GOOD: {result}')
        return result

    # (b) 모델 1순위가 good이고 규칙상 정상이면 good
    top_class = classes[int(np.argmax(combined))]
    if top_class == 'good' and back_ok and heel_ok:
        if verbose: print(f'  → TOP-IS-GOOD + RULE-OK')
        return ['good']

    # (c) 다중 라벨 검출
    detected = []
    if bad_back_score >= MULTI_LABEL_THR:
        detected.append('bad_back')
    # heel은 모델 점수 + 규칙 둘 다 만족해야 추가 (더 보수적)
    if bad_heel_score >= MULTI_LABEL_THR and heel_lifted:
        detected.append('bad_heel')

    # 규칙 보강 — false negative (good_score 낮고 명백 위반일 때)
    if good_score < 0.40:
        if back_bent and 'bad_back' not in detected:
            detected.append('bad_back')
        # heel 규칙 보강은 model_score도 어느 정도 있어야 (단독 규칙 추가 X)
        if heel_lifted and bad_heel_score > 0.30 and 'bad_heel' not in detected:
            detected.append('bad_heel')

    # 규칙 보강 — false positive 제거
    if 'bad_back' in detected and back_ok and bad_back_score < CONFIDENCE_THR:
        detected.remove('bad_back')
    # heel은 더 적극적으로 제거: heel_ok이거나 heel_lifted가 False면 제거
    if 'bad_heel' in detected:
        if heel_ok or (not heel_lifted and bad_heel_score < 0.70):
            detected.remove('bad_heel')

    result = detected if detected else ['good']
    if verbose: print(f'  → 최종: {result}')
    return result


print('예측 함수 정의 완료! (발뒤꿈치 false positive 차단 강화)')


---
### 3. 각도 계산 및 특징 추출 함수

In [ ]:
def calc_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba, bc = a - b, c - b
    cos = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    return np.degrees(np.arccos(np.clip(cos, -1.0, 1.0)))


def calc_vertical_angle(top, bottom):
    dx = top[0] - bottom[0]
    dy = top[1] - bottom[1]
    return np.degrees(np.arctan2(abs(dx), abs(dy)))


def extract_features_from_lm(lm):
    """14개 피처. 거리값은 대퇴골 길이로 정규화."""
    def pt(i): return [lm[i].x, lm[i].y]

    femur_L = np.sqrt((lm[23].x - lm[25].x) ** 2 + (lm[23].y - lm[25].y) ** 2)
    femur_R = np.sqrt((lm[24].x - lm[26].x) ** 2 + (lm[24].y - lm[26].y) ** 2)
    ref_length = (femur_L + femur_R) / 2 + 1e-6

    knee_L  = calc_angle(pt(23), pt(25), pt(27))
    knee_R  = calc_angle(pt(24), pt(26), pt(28))
    hip_L   = calc_angle(pt(11), pt(23), pt(25))
    hip_R   = calc_angle(pt(12), pt(24), pt(26))
    ankle_L = calc_angle(pt(25), pt(27), pt(29))
    ankle_R = calc_angle(pt(26), pt(28), pt(30))

    cave_L = (lm[25].x - lm[27].x) / ref_length
    cave_R = (lm[26].x - lm[28].x) / ref_length
    heel_lift_L = (lm[31].y - lm[29].y) / ref_length
    heel_lift_R = (lm[32].y - lm[30].y) / ref_length

    torso_L = calc_vertical_angle(pt(11), pt(23))
    torso_R = calc_vertical_angle(pt(12), pt(24))
    sym_knee = abs(knee_L - knee_R)

    knee_y_avg = (lm[25].y + lm[26].y) / 2
    hip_y_avg  = (lm[23].y + lm[24].y) / 2
    knee_depth = (knee_y_avg - hip_y_avg) / ref_length

    return [
        knee_L, knee_R, hip_L, hip_R, ankle_L, ankle_R,
        cave_L, cave_R, heel_lift_L, heel_lift_R,
        torso_L, torso_R, sym_knee, knee_depth,
    ]


def extract_features_from_image(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return None
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    result = mp_pose_img.process(img_rgb)
    if not result.pose_landmarks:
        return None
    return extract_features_from_lm(result.pose_landmarks.landmark)


def get_avg_knee_angle(lm):
    """Visibility 가중 무릎 각도 평균 — 측면 촬영 강건"""
    def pt(i): return [lm[i].x, lm[i].y]
    left  = calc_angle(pt(23), pt(25), pt(27))
    right = calc_angle(pt(24), pt(26), pt(28))
    v_L = lm[25].visibility
    v_R = lm[26].visibility
    return (left * v_L + right * v_R) / (v_L + v_R + 1e-6)


print('함수 정의 완료!')


---
### 4. 모델 로드 및 피드백 설정

In [ ]:
# 02_train.py 로 학습한 모델 (노트북은 notebooks/ 에서 실행한다고 가정)
MODEL_PATH = os.path.join('..', 'models', 'rf_model.pkl')

with open(MODEL_PATH, 'rb') as f:
    rf_ckpt = pickle.load(f)

rf_model = rf_ckpt['model']
rf_le    = rf_ckpt['label_encoder']
FEAT     = rf_ckpt['features']

print('모델 로드 완료!')
print('클래스 순서:', rf_le.classes_)

In [ ]:
# ============================================
# 디자인 시스템 (BGR — OpenCV 표준)
# ============================================
PALETTE = {
    'bg':         (28,  18,  10),     # #0A121C 딥 네이비 블랙
    'surface':    (52,  37,  22),     # #16253A
    'surface_lt': (89,  67,  44),     # #2C4359
    'text':       (252, 250, 248),    # #F8FAFC
    'text_dim':   (184, 163, 148),    # #94A3B8
    'accent':     (255, 229,   0),    # #00E5FF 네온 시안
    'success':    (129, 185,  16),    # #10B981 에메랄드
    'warning':    (36,  191, 251),    # #FBBF24 앰버
    'danger':     (68,   68, 239),    # #EF4444 레드
    'critical':   (97,   31, 196),    # #C41F61 마젠타
}

LABEL_COLOR = {
    'good':     PALETTE['success'],
    'shallow':  PALETTE['warning'],
    'bad_heel': PALETTE['danger'],
    'bad_back': PALETTE['critical'],
}

FEEDBACK_KO = {
    'good':     '올바른 자세입니다',
    'bad_back': '허리가 굽었습니다',
    'bad_heel': '발뒤꿈치가 들렸습니다',
    'shallow':  '더 깊이 앉아주세요',
}

FEEDBACK_SUB = {
    'good':     '척추 정렬 / 발바닥 안정',
    'bad_back': '척추를 곧게 펴고 다시 시도',
    'bad_heel': '발바닥 전체를 지면에 붙이세요',
    'shallow':  '엉덩이를 무릎 높이까지 낮추세요',
}


def format_feedback(detected):
    """detected: list[str] → (메인 메시지, 보조 메시지, 색상, 상태)"""
    if not detected or detected == ['good']:
        return FEEDBACK_KO['good'], FEEDBACK_SUB['good'], LABEL_COLOR['good'], 'good'
    if 'shallow' in detected:
        return FEEDBACK_KO['shallow'], FEEDBACK_SUB['shallow'], LABEL_COLOR['shallow'], 'warn'

    bad_items = [d for d in detected if d in ('bad_back', 'bad_heel')]
    if len(bad_items) >= 2:
        return '허리 + 발뒤꿈치 동시 교정', \
               '척추를 펴고 발바닥을 붙이세요', \
               LABEL_COLOR['bad_back'], 'bad'
    d = bad_items[0]
    return FEEDBACK_KO[d], FEEDBACK_SUB[d], LABEL_COLOR[d], 'bad'


print('피드백 + 디자인 시스템 설정 완료!')


---
### 5. 모던 UI 렌더링 헬퍼

**디자인 시스템**
- 컬러: 다크 네이비 + 네온 액센트 (피트니스 앱 트렌드)
- 컴포넌트: 둥근 카드, 반투명 오버레이, 그라데이션
- 폰트: PIL `Malgun Gothic`으로 한글 정상 출력 (OpenCV `putText` 한글 깨짐 해결)
- 레이아웃: 좌상단 각도 카드 / 우상단 카운터 카드 / 하단 피드백 카드


In [ ]:
# ============================================
# UI 렌더링 헬퍼 (PIL + OpenCV 하이브리드)
# ============================================
import platform

if platform.system() == 'Windows':
    FONT_PATH_BOLD   = 'C:/Windows/Fonts/malgunbd.ttf'
    FONT_PATH_NORMAL = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':
    FONT_PATH_BOLD   = '/System/Library/Fonts/AppleSDGothicNeo.ttc'
    FONT_PATH_NORMAL = '/System/Library/Fonts/AppleSDGothicNeo.ttc'
else:
    FONT_PATH_BOLD   = '/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf'
    FONT_PATH_NORMAL = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

_FONT_CACHE = {}


def get_font(size, bold=False):
    key = (size, bold)
    if key not in _FONT_CACHE:
        try:
            path = FONT_PATH_BOLD if bold else FONT_PATH_NORMAL
            _FONT_CACHE[key] = ImageFont.truetype(path, size)
        except Exception:
            _FONT_CACHE[key] = ImageFont.load_default()
    return _FONT_CACHE[key]


def draw_rounded_rect(img, pt1, pt2, color, radius=14, alpha=0.85):
    """반투명 둥근 사각형"""
    overlay = img.copy()
    x1, y1 = pt1
    x2, y2 = pt2
    cv2.rectangle(overlay, (x1 + radius, y1), (x2 - radius, y2), color, -1)
    cv2.rectangle(overlay, (x1, y1 + radius), (x2, y2 - radius), color, -1)
    cv2.circle(overlay, (x1 + radius, y1 + radius), radius, color, -1)
    cv2.circle(overlay, (x2 - radius, y1 + radius), radius, color, -1)
    cv2.circle(overlay, (x1 + radius, y2 - radius), radius, color, -1)
    cv2.circle(overlay, (x2 - radius, y2 - radius), radius, color, -1)
    cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0, img)
    return img


def draw_accent_bar(img, pt1, pt2, color, thickness=4):
    """좌측 액센트 바"""
    x1, y1 = pt1
    cv2.rectangle(img, (x1, y1), (x1 + thickness, pt2[1]), color, -1)
    return img


def put_text(img, text, position, size=22, bold=False, color=(255, 255, 255),
             align='left'):
    """PIL로 한글 텍스트 그리기"""
    font = get_font(size, bold)
    img_pil = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(img_pil)
    if align in ('center', 'right'):
        bbox = draw.textbbox((0, 0), text, font=font)
        text_w = bbox[2] - bbox[0]
        x, y = position
        if align == 'center':
            x -= text_w // 2
        else:
            x -= text_w
        position = (x, y)
    rgb_color = (color[2], color[1], color[0])
    draw.text(position, text, font=font, fill=rgb_color)
    return cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)


# ============================================
# 아이콘 — OpenCV 도형으로 직접 그리기 (폰트 의존성 X)
# ============================================
def draw_icon_check(img, center, color, size=14, thickness=4):
    """체크마크"""
    x, y = center
    pts = [(x - size, y), (x - size // 4, y + size // 2),
           (x + size, y - size // 2)]
    cv2.line(img, pts[0], pts[1], color, thickness, cv2.LINE_AA)
    cv2.line(img, pts[1], pts[2], color, thickness, cv2.LINE_AA)
    return img


def draw_icon_x(img, center, color, size=14, thickness=4):
    """X 마크"""
    x, y = center
    cv2.line(img, (x - size, y - size), (x + size, y + size),
             color, thickness, cv2.LINE_AA)
    cv2.line(img, (x - size, y + size), (x + size, y - size),
             color, thickness, cv2.LINE_AA)
    return img


def draw_icon_warn(img, center, color, size=14, thickness=4):
    """! 마크"""
    x, y = center
    cv2.line(img, (x, y - size), (x, y + size // 3),
             color, thickness, cv2.LINE_AA)
    cv2.circle(img, (x, y + size - 2), 3, color, -1, cv2.LINE_AA)
    return img


def draw_icon_dot(img, center, color, size=10):
    """● 마크 (대기 상태)"""
    cv2.circle(img, center, size // 2, color, -1, cv2.LINE_AA)
    return img


def draw_icon_pulse(img, center, color, size=14, thickness=4):
    """⚡ 마크 (감지 중) — 번개 모양"""
    x, y = center
    pts = np.array([
        [x - size // 2, y - size],
        [x + size // 3, y - 2],
        [x - size // 3, y + 2],
        [x + size // 2, y + size],
    ], dtype=np.int32)
    cv2.polylines(img, [pts], False, color, thickness, cv2.LINE_AA)
    return img


def draw_icon_by_status(img, center, status, color, size=14):
    """상태에 따라 적절한 아이콘"""
    if status == 'good':
        return draw_icon_check(img, center, (255, 255, 255), size)
    elif status == 'warn':
        return draw_icon_warn(img, center, (255, 255, 255), size)
    elif status == 'bad':
        return draw_icon_x(img, center, (255, 255, 255), size)
    elif status == 'detecting':
        return draw_icon_pulse(img, center, color, size)
    else:
        return draw_icon_dot(img, center, (255, 255, 255), size)


def draw_phase_indicator(img, x, y, phase):
    """페이즈 표시"""
    if phase == 'SQUATTING':
        color = PALETTE['accent']
        label = 'SQUATTING'
    else:
        color = PALETTE['text_dim']
        label = 'IDLE'
    cv2.circle(img, (x, y), 5, color, -1, cv2.LINE_AA)
    cv2.circle(img, (x, y), 8, color, 1, cv2.LINE_AA)
    img = put_text(img, label, (x + 16, y - 11), size=14, bold=True,
                   color=PALETTE['text'])
    return img


def draw_progress_bar(img, pt1, pt2, ratio, color):
    """진행 바"""
    cv2.rectangle(img, pt1, pt2, PALETTE['surface_lt'], -1)
    fill_x = int(pt1[0] + (pt2[0] - pt1[0]) * max(0.0, min(1.0, ratio)))
    cv2.rectangle(img, pt1, (fill_x, pt2[1]), color, -1)
    return img


print('UI 헬퍼 정의 완료! (아이콘: OpenCV 도형, 폰트: PIL)')


---
### 6. 영상 테스트 — 모던 UI

**화면 구성**
- 좌상단: 무릎 각도 카드 (실시간) + 페이즈 인디케이터
- 우상단: 스쿼트 카운터 + 폼 스코어 (good 비율)
- 하단: 피드백 카드 (메인 + 보조 메시지)
- 카운터 진행 바: 스쿼트 깊이 시각화


In [ ]:
def predict_video(video_path, loop=False):
    cap = cv2.VideoCapture(video_path)

    phase            = 'IDLE'
    feat_buf         = []
    feedback_main    = ''
    feedback_sub     = ''
    feedback_color   = PALETTE['text_dim']
    feedback_status  = 'idle'
    result_countdown = 0
    last_detected    = None

    enter_count = exit_count = 0

    rep_count  = 0
    good_count = 0
    history    = []

    while True:
        ret, frame = cap.read()
        if not ret:
            if loop:
                cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
                phase = 'IDLE'
                feat_buf = []
                enter_count = exit_count = 0
                # ── 루프 재시작 시에도 q 체크 ──
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break
                continue
            else:
                print('영상 재생 완료')
                break

        h, w = frame.shape[:2]
        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        res = mp_pose_video.process(img_rgb)

        knee_angle = None
        if res.pose_landmarks:
            lm = res.pose_landmarks.landmark

            mp_draw.draw_landmarks(
                frame, res.pose_landmarks,
                mp.solutions.pose.POSE_CONNECTIONS,
                landmark_drawing_spec=mp_draw.DrawingSpec(
                    color=PALETTE['accent'], thickness=2, circle_radius=3),
                connection_drawing_spec=mp_draw.DrawingSpec(
                    color=(180, 180, 180), thickness=2),
            )

            knee_angle = get_avg_knee_angle(lm)

            if phase == 'IDLE':
                if knee_angle < SQUAT_ENTER:
                    enter_count += 1
                    if enter_count >= N_CONSEC_FRAMES:
                        phase = 'SQUATTING'
                        feat_buf = []
                        enter_count = 0
                else:
                    enter_count = 0
            elif phase == 'SQUATTING':
                feat_buf.append(extract_features_from_lm(lm))
                if knee_angle >= SQUAT_EXIT:
                    exit_count += 1
                    if exit_count >= N_CONSEC_FRAMES:
                        if len(feat_buf) >= MIN_SQUAT_FRAMES:
                            last_detected = predict_from_features(feat_buf)
                            feedback_main, feedback_sub, feedback_color, \
                                feedback_status = format_feedback(last_detected)
                            result_countdown = 90
                            rep_count += 1
                            if last_detected == ['good']:
                                good_count += 1
                            history.append(last_detected)
                            print(f'[Rep {rep_count}] 예측: {last_detected}')
                        phase = 'IDLE'
                        feat_buf = []
                        exit_count = 0
                else:
                    exit_count = 0

        # ============================================
        # UI 렌더링
        # ============================================
        gradient = np.zeros_like(frame)
        gradient[:140] = PALETTE['bg']
        cv2.addWeighted(gradient, 0.4, frame, 1.0, 0, frame)

        card1_pt1 = (20, 20)
        card1_pt2 = (260, 110)
        draw_rounded_rect(frame, card1_pt1, card1_pt2, PALETTE['surface'],
                          radius=16, alpha=0.92)
        accent_color = feedback_color if result_countdown > 0 else PALETTE['accent']
        draw_accent_bar(frame, card1_pt1, card1_pt2, accent_color)

        frame = put_text(frame, 'KNEE ANGLE', (40, 30),
                         size=11, bold=True, color=PALETTE['text_dim'])
        angle_text = f'{knee_angle:.1f}°' if knee_angle is not None else '— —'
        frame = put_text(frame, angle_text, (40, 47),
                         size=32, bold=True, color=PALETTE['text'])
        frame = draw_phase_indicator(frame, 170, 92, phase)

        card2_pt1 = (w - 260, 20)
        card2_pt2 = (w - 20, 110)
        draw_rounded_rect(frame, card2_pt1, card2_pt2, PALETTE['surface'],
                          radius=16, alpha=0.92)
        draw_accent_bar(frame, card2_pt1, card2_pt2, PALETTE['accent'])

        frame = put_text(frame, 'REPS', (w - 240, 30),
                         size=11, bold=True, color=PALETTE['text_dim'])
        frame = put_text(frame, f'{rep_count:02d}', (w - 240, 47),
                         size=32, bold=True, color=PALETTE['text'])

        if rep_count > 0:
            form_score = good_count / rep_count * 100
            score_color = (PALETTE['success'] if form_score >= 80
                           else PALETTE['warning'] if form_score >= 50
                           else PALETTE['danger'])
            frame = put_text(frame, 'FORM', (w - 130, 30),
                             size=11, bold=True, color=PALETTE['text_dim'])
            frame = put_text(frame, f'{form_score:.0f}%', (w - 130, 47),
                             size=28, bold=True, color=score_color)
        else:
            frame = put_text(frame, 'FORM', (w - 130, 30),
                             size=11, bold=True, color=PALETTE['text_dim'])
            frame = put_text(frame, '—', (w - 130, 47),
                             size=28, bold=True, color=PALETTE['text_dim'])

        if history:
            dot_x = w - 240
            dot_y = 93
            for i, h_item in enumerate(history[-5:]):
                color = (PALETTE['success'] if h_item == ['good']
                         else PALETTE['warning'] if 'shallow' in h_item
                         else PALETTE['danger'])
                cv2.circle(frame, (dot_x + i * 16, dot_y), 5, color, -1,
                           cv2.LINE_AA)

        if phase == 'SQUATTING' and knee_angle is not None:
            depth_ratio = max(0, min(1, (SQUAT_ENTER - knee_angle)
                                        / (SQUAT_ENTER - SQUAT_BOTTOM)))
            bar_y = 130
            draw_progress_bar(frame, (20, bar_y), (w - 20, bar_y + 6),
                              depth_ratio, PALETTE['accent'])
            frame = put_text(frame, f'DEPTH  {depth_ratio*100:.0f}%',
                             (20, bar_y + 12),
                             size=11, bold=True, color=PALETTE['text_dim'])

        card3_h = 110
        card3_pt1 = (20, h - card3_h - 20)
        card3_pt2 = (w - 20, h - 20)

        if result_countdown > 0:
            draw_rounded_rect(frame, card3_pt1, card3_pt2, PALETTE['surface'],
                              radius=20, alpha=0.94)
            draw_accent_bar(frame, card3_pt1, card3_pt2, feedback_color,
                            thickness=6)
            icon_x = card3_pt1[0] + 50
            icon_y = (card3_pt1[1] + card3_pt2[1]) // 2
            cv2.circle(frame, (icon_x, icon_y), 26, feedback_color, -1,
                       cv2.LINE_AA)
            draw_icon_by_status(frame, (icon_x, icon_y), feedback_status,
                                (255, 255, 255), size=14)
            frame = put_text(frame, feedback_main, (icon_x + 50, icon_y - 24),
                             size=22, bold=True, color=PALETTE['text'])
            frame = put_text(frame, feedback_sub, (icon_x + 50, icon_y + 4),
                             size=14, bold=False, color=PALETTE['text_dim'])
            frame = put_text(frame, f'#{rep_count:02d}',
                             (card3_pt2[0] - 30, icon_y - 18),
                             size=24, bold=True, color=feedback_color,
                             align='right')
            result_countdown -= 1

        elif phase == 'SQUATTING':
            draw_rounded_rect(frame, card3_pt1, card3_pt2, PALETTE['surface'],
                              radius=20, alpha=0.92)
            draw_accent_bar(frame, card3_pt1, card3_pt2, PALETTE['accent'],
                            thickness=6)
            icon_x = card3_pt1[0] + 50
            icon_y = (card3_pt1[1] + card3_pt2[1]) // 2
            cv2.circle(frame, (icon_x, icon_y), 26, PALETTE['accent'], -1,
                       cv2.LINE_AA)
            draw_icon_pulse(frame, (icon_x, icon_y), PALETTE['bg'], size=14)
            frame = put_text(frame, '동작 감지 중', (icon_x + 50, icon_y - 24),
                             size=22, bold=True, color=PALETTE['text'])
            frame = put_text(frame, f'{len(feat_buf)} frames captured',
                             (icon_x + 50, icon_y + 4),
                             size=14, bold=False, color=PALETTE['text_dim'])

        else:
            draw_rounded_rect(frame, card3_pt1, card3_pt2, PALETTE['surface'],
                              radius=20, alpha=0.85)
            draw_accent_bar(frame, card3_pt1, card3_pt2, PALETTE['text_dim'],
                            thickness=6)
            icon_x = card3_pt1[0] + 50
            icon_y = (card3_pt1[1] + card3_pt2[1]) // 2
            cv2.circle(frame, (icon_x, icon_y), 26, PALETTE['surface_lt'], -1,
                       cv2.LINE_AA)
            draw_icon_dot(frame, (icon_x, icon_y), PALETTE['text_dim'], size=12)
            frame = put_text(frame, '준비 자세를 취해주세요',
                             (icon_x + 50, icon_y - 24),
                             size=22, bold=True, color=PALETTE['text'])
            frame = put_text(frame, '무릎 각도 155° 미만이 될 때 시작',
                             (icon_x + 50, icon_y + 4),
                             size=14, bold=False, color=PALETTE['text_dim'])

        frame = put_text(frame, 'SQUAT FORM AI',
                         (w - 30, h - card3_h - 38),
                         size=10, bold=True, color=PALETTE['text_dim'],
                         align='right')

        cv2.imshow('Squat Form AI', frame)

        # ── q 또는 ESC로 종료 ──
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or key == 27:
            print('사용자가 종료했습니다.')
            break

    cap.release()
    cv2.destroyAllWindows()
    return last_detected


# ── 실행 ──
# 영상 파일 경로를 넣거나, 웹캠을 쓰려면 VIDEO_SOURCE = 0
# 창에서 q 또는 ESC로 종료
VIDEO_SOURCE = os.path.join('..', 'test_videos', 'sample.mp4')
result = predict_video(VIDEO_SOURCE, loop=False)
print(f'최종 예측: {result}')